In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# 1. Input data (Function 4)
# ============================================================
X_train = np.array([
    [8.96981054e-01, 7.25627970e-01, 1.75404309e-01, 7.01694369e-01],
    [8.89356396e-01, 4.99587855e-01, 5.39268858e-01, 5.08783439e-01],
    [2.50946243e-01, 3.36931305e-02, 1.45380025e-01, 4.94932421e-01],
    [3.46962061e-01, 6.25040024e-03, 7.60563606e-01, 6.13023557e-01],
    [1.24871181e-01, 1.29770193e-01, 3.84400483e-01, 2.87076101e-01],
    [8.01302707e-01, 5.00231094e-01, 7.06644560e-01, 1.95102841e-01],
    [2.47708262e-01, 6.04454273e-02, 4.21863451e-02, 4.41324251e-01],
    [7.46702242e-01, 7.57091504e-01, 3.69353060e-01, 2.06566281e-01],
    [4.00665027e-01, 7.25742511e-02, 8.86768254e-01, 2.43842290e-01],
    [6.26070596e-01, 5.86751259e-01, 4.38805782e-01, 7.78857694e-01],
    [9.57135293e-01, 5.97644383e-01, 7.66113852e-01, 7.76209905e-01],
    [7.32812426e-01, 1.45249979e-01, 4.76812718e-01, 1.33365734e-01],
    [6.55115479e-01, 7.23918269e-02, 6.87151746e-01, 8.15165642e-02],
    [2.19734429e-01, 8.32031335e-01, 4.82864162e-01, 8.25692306e-02],
    [4.88594190e-01, 2.11965096e-01, 9.39177907e-01, 3.76191726e-01],
    [1.67130486e-01, 8.76554558e-01, 2.17239545e-01, 9.59800985e-01],
    [2.16911188e-01, 1.66085829e-01, 2.41372256e-01, 7.70062476e-01],
    [3.87487837e-01, 8.04532258e-01, 7.51795483e-01, 7.23827439e-01],
    [9.85621893e-01, 6.66932679e-01, 1.56783283e-01, 8.56534801e-01],
    [3.78248285e-02, 6.64853346e-01, 1.61982175e-01, 2.53923780e-01],
    [6.83486385e-01, 9.02770103e-01, 3.35419826e-01, 9.99482561e-01],
    [1.70347305e-01, 7.56959083e-01, 2.76520486e-01, 5.31231498e-01],
    [8.59656919e-01, 9.19592322e-01, 2.06138728e-01, 9.77968310e-02],
    [2.82138368e-01, 5.05986912e-01, 5.30530843e-01, 9.63016230e-02],
    [3.26075785e-01, 4.72366904e-01, 4.53191996e-01, 1.05887338e-01],
    [9.48389362e-01, 8.94513008e-01, 8.51637817e-01, 5.52196286e-01],
    [6.64955390e-01, 4.65662767e-02, 1.16777469e-01, 7.93717780e-01],
    [5.77765614e-01, 4.28771742e-01, 4.25825867e-01, 2.49007415e-01],
    [7.38613014e-01, 4.82102634e-01, 7.09366443e-01, 5.03970014e-01],
    [8.54810797e-01, 4.93964620e-01, 7.35309975e-01, 8.08092013e-01],
    [1.08562100e+00, 1.01959200e+00, 1.03917700e+00, 1.09948200e+00],
    [1.00000000e-06, 1.00000000e-06, 1.24558000e-01, 1.00000000e-06],
    [8.66175000e-01, 6.01115000e-01, 7.08072000e-01, 2.05850000e-02],
    [1.45904000e-01, 5.36548000e-01, 6.01400000e-01, 1.90500000e-02],
])

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.6049343 , -22.78219342, -22.19421279, -13.36310565,
])

# ============================================================
# 2. Basic info: current best point
# ============================================================
# maximisation: best is largest y
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)

# ============================================================
# 3. Normalise inputs to [0,1] (per feature)
# ============================================================
X_min = X_train.min(axis=0)
X_max = X_train.max(axis=0)

X_scaled = (X_train - X_min) / (X_max - X_min + 1e-12)

# ============================================================
# 4. Neural network surrogate with MC Dropout
# ============================================================
class MLPDropout(nn.Module):
    """
    Simple 2-hidden-layer MLP with dropout.
    Dropout is kept ON at prediction time for approximate Bayesian uncertainty.
    """
    def __init__(self, input_dim=4, hidden_dim=64, p_dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

# Seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).unsqueeze(-1)

model = MLPDropout(input_dim=4, hidden_dim=64, p_dropout=0.1).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ============================================================
# 5. Train NN surrogate
# ============================================================
n_epochs = 2000  # you can increase if you want more training

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = criterion(preds, y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 400 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} - Train MSE: {loss.item():.4f}")

# ============================================================
# 6. MC Dropout prediction to get mean & std
# ============================================================
def mc_predict(model, X, n_samples=100):
    """
    MC Dropout prediction to estimate predictive mean and std.
    X: torch tensor [N, D] in scaled space.
    """
    model.train()  # keep dropout active
    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            preds.append(model(X).squeeze(-1))
    preds = torch.stack(preds, dim=0)  # [S, N]
    mean = preds.mean(dim=0)
    std = preds.std(dim=0) + 1e-9
    return mean, std

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.01):
    """
    EI for maximisation with normal approximation.
    """
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    ei = torch.where(std > 0, ei, torch.zeros_like(ei))
    return ei

def probability_of_improvement(mean, std, best_y, xi=0.0):
    """
    PI = P(f(x) > best_y + xi)
    """
    imp = mean - best_y - xi
    Z = imp / std
    pi = normal.cdf(Z)
    return pi

# ============================================================
# 7. Propose next query point using EI
# ============================================================
def propose_next_point(model, X_min, X_max, n_candidates=5000):
    # Sample candidates uniformly in scaled [0,1]^4
    candidates_scaled = np.random.rand(n_candidates, 4).astype(np.float32)
    X_cand_tensor = torch.tensor(candidates_scaled, dtype=torch.float32, device=device)

    mean, std = mc_predict(model, X_cand_tensor, n_samples=100)

    best_y = current_best_y
    ei = expected_improvement(mean, std, best_y)
    pi = probability_of_improvement(mean, std, best_y)

    best_idx = int(torch.argmax(ei).item())
    next_x_scaled = candidates_scaled[best_idx]

    # Unscale back to original space
    next_x = X_min + next_x_scaled * (X_max - X_min)

    next_mean = float(mean[best_idx].item())
    next_std = float(std[best_idx].item())
    next_ei = float(ei[best_idx].item())
    next_pi = float(pi[best_idx].item())

    return next_x, next_mean, next_std, next_ei, next_pi

next_x, next_mean, next_std, next_ei, next_pi = propose_next_point(
    model, X_min, X_max, n_candidates=5000
)

# ============================================================
# 8. Report: current best vs next query, probabilities & reasoning
# ============================================================
print("\n================ RESULTS ================")
print("Current best X:", current_best_x)
print("Current best y:", current_best_y)
print("\nProposed next query point X*:", next_x)
print("Predicted mean at X*:", next_mean)
print("Predictive std at X*:", next_std)
print("Expected Improvement at X*:", next_ei)
print("Probability of Improvement P(f(X*) > f_best):", next_pi)

# Human-readable reasoning
improvement_margin = next_mean - current_best_y
print("\nReasoning:")
print(f"- The neural-network surrogate predicts an output of {next_mean:.3f} at X*,")
print(f"  compared to the current best of {current_best_y:.3f}, so the expected gain is {improvement_margin:.3f}.")
print(f"- The predictive uncertainty (std) at X* is {next_std:.3f}.")
print(f"- Combining mean and uncertainty gives an Expected Improvement (EI) of {next_ei:.3f},")
print(f"  and a Probability of Improvement of {next_pi:.3f} (~{next_pi*100:.1f}% chance of beating the current best).")
print("- This point lies in a region where the model predicts relatively high performance")
print("  but still has non-zero uncertainty (from dropout), so it balances exploitation and exploration.")


Current best index: 27
Current best X: [0.57776561 0.42877174 0.42582587 0.24900742]
Current best y: -4.02554228
Epoch 400/2000 - Train MSE: 34.7091
Epoch 800/2000 - Train MSE: 8.3599
Epoch 1200/2000 - Train MSE: 4.4565
Epoch 1600/2000 - Train MSE: 4.1131
Epoch 2000/2000 - Train MSE: 3.2062

================ RESULTS ================
Current best X: [0.57776561 0.42877174 0.42582587 0.24900742]
Current best y: -4.02554228

Proposed next query point X*: [0.3562936  0.44252323 0.13052014 0.24255966]
Predicted mean at X*: -9.711309432983398
Predictive std at X*: 1.5682331323623657
Expected Improvement at X*: 5.3823343478143215e-05
Probability of Improvement P(f(X*) > f_best): 0.00014415383338928223

Reasoning:
- The neural-network surrogate predicts an output of -9.711 at X*,
  compared to the current best of -4.026, so the expected gain is -5.686.
- The predictive uncertainty (std) at X* is 1.568.
- Combining mean and uncertainty gives an Expected Improvement (EI) of 0.000,
  and a Probab